In [ ]:
import pandas as pd
import numpy as np

In [2]:
# Load Dataset
df_train = pd.read_csv('/content/Training.csv')
df_test  = pd.read_csv('/content/Testing.csv')


In [3]:
df_train.head()

,itching,skin_rash,nodal_skin_eruptions,continuous_sneezing,shivering,chills,joint_pain,stomach_pain,acidity,ulcers_on_tongue,...,scurring,skin_peeling,silver_like_dusting,small_dents_in_nails,inflammatory_nails,blister,red_sore_around_nose,yellow_crust_ooze,prognosis,Unnamed: 133
0,1,1,1,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,Fungal infection,NaN
1,0,1,1,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,Fungal infection,NaN
2,1,0,1,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,Fungal infection,NaN
3,1,1,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,Fungal infection,NaN
4,1,1,1,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,Fungal infection,NaN


In [ ]:
df_test.head()

,itching,skin_rash,nodal_skin_eruptions,continuous_sneezing,shivering,chills,joint_pain,stomach_pain,acidity,ulcers_on_tongue,...,blackheads,scurring,skin_peeling,silver_like_dusting,small_dents_in_nails,inflammatory_nails,blister,red_sore_around_nose,yellow_crust_ooze,prognosis
0,1,1,1,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,Fungal infection
1,0,0,0,1,1,1,0,0,0,0,...,0,0,0,0,0,0,0,0,0,Allergy
2,0,0,0,0,0,0,0,1,1,1,...,0,0,0,0,0,0,0,0,0,GERD
3,1,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,Chronic cholestasis
4,1,1,0,0,0,0,0,1,0,0,...,0,0,0,0,0,0,0,0,0,Drug Reaction


In [4]:
df_test.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 42 entries, 0 to 41
Columns: 133 entries, itching to prognosis
dtypes: int64(132), object(1)
memory usage: 43.8+ KB


In [5]:
print("Train shape:", df_train.shape)
print("Test shape: ", df_test.shape)

Train shape: (4920, 134)
Test shape:  (42, 133)


In [6]:
print("Columns:", df_train.columns.tolist()[-5:])  # last 5 cols
print("\nTarget sample:\n", df_train['prognosis'].value_counts().head(5))


Columns: ['blister', 'red_sore_around_nose', 'yellow_crust_ooze', 'prognosis', 'Unnamed: 133']

Target sample:
 prognosis
Fungal infection       120
Allergy                120
GERD                   120
Chronic cholestasis    120
Drug Reaction          120
Name: count, dtype: int64


In [7]:
# Clean (last column sometimes unnamed garbage)
df_train = df_train.loc[:, ~df_train.columns.str.contains('^Unnamed')]
df_test  = df_test.loc[:,  ~df_test.columns.str.contains('^Unnamed')]

In [8]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, accuracy_score
from sklearn.preprocessing import LabelEncoder

In [9]:
# Features & Target
le = LabelEncoder()
y_train = le.fit_transform(df_train['prognosis'])
y_test  = le.transform(df_test['prognosis'])

X_train = df_train.drop(columns=['prognosis'])
X_test  = df_test.drop(columns=['prognosis'])

print(f"\nFeatures : {X_train.shape[1]} symptoms")
print(f"Classes  : {len(le.classes_)} diseases")


Features : 132 symptoms
Classes  : 41 diseases


In [13]:
# Model
model = RandomForestClassifier(
    n_estimators=200,
    random_state=42,
    n_jobs=-1
)
model.fit(X_train, y_train)

RandomForestClassifier(n_estimators=200, n_jobs=-1, random_state=42)

In [14]:
# Evaluate
y_pred = model.predict(X_test)
acc = accuracy_score(y_test, y_pred)
print(f"\n Test Accuracy: {acc:.4f}")
print("\n", classification_report(y_test, y_pred, target_names=le.classes_))



 Test Accuracy: 0.9762

                                          precision    recall  f1-score   support

(vertigo) Paroymsal  Positional Vertigo       1.00      1.00      1.00         1
                                   AIDS       1.00      1.00      1.00         1
                                   Acne       1.00      1.00      1.00         1
                    Alcoholic hepatitis       1.00      1.00      1.00         1
                                Allergy       1.00      1.00      1.00         1
                              Arthritis       1.00      1.00      1.00         1
                       Bronchial Asthma       1.00      1.00      1.00         1
                   Cervical spondylosis       1.00      1.00      1.00         1
                            Chicken pox       1.00      1.00      1.00         1
                    Chronic cholestasis       1.00      1.00      1.00         1
                            Common Cold       1.00      1.00      1.00         1
 

In [15]:
# Top 10 Important Symptoms
feat_imp = pd.Series(model.feature_importances_, index=X_train.columns)
print("\nTop 10 Most Important Symptoms:")
print(feat_imp.nlargest(10))


Top 10 Most Important Symptoms:
muscle_pain          0.019514
itching              0.016854
dark_urine           0.015853
high_fever           0.015595
altered_sensorium    0.014875
fatigue              0.014580
nausea               0.014471
mild_fever           0.014310
yellowing_of_eyes    0.014072
family_history       0.013757
dtype: float64
